# Generate the Nestle 1904 Greek New Testament documentation
This notebook builds the complete **N1904** example using the current context-first Doc4CF workflow. Run the cells in order from a Doc4CF checkout with a Python 3.13+ kernel, Git, and network access for installation/source download.

Source: [CenterBLC/N1904](https://github.com/CenterBLC/N1904), TF version **1.0.0**, pinned revision `3f53a5a9075cbca5c3001a000a7b004cdee251b6`. The notebook checks out only that revision in its own workspace. Generated files and caches stay in ignored local directories until the optional publication step.

LLM enrichment is off by default. Set `USE_LLM = True` to create and review dataset/family context artifacts before generating context-aware feature descriptions. It uses **nvidia/nemotron-3.5-super-vl:free** by default and includes the N1904 editorial notes as background during feature generation. New model responses may differ from checked-in prose. All measured corpus pages remain based on the pinned dataset.

## 1. Locate and install Doc4CF
Select the Python 3.13+ kernel before running this cell. The editable install uses the code in this checkout; restart the kernel if your notebook environment asks you to.

In [ ]:
from pathlib import Path
import sys
import subprocess
import shutil
import os
import json

assert sys.version_info >= (3, 13), "Select a Python 3.13+ kernel."
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "doc4cf").is_dir()), None)
assert ROOT is not None, "Open this notebook inside the Doc4CF checkout."
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT)], check=True)
assert shutil.which("git"), "Install Git and restart the kernel."

## 2. Fetch the pinned corpus
Only the base TF version is used, without optional annotation modules. A dedicated clone under `.testdata/example-sources` keeps source data separate from generated HTML. Existing clones are reused only if clean; the revision check prevents silent changes to the source.

In [ ]:
NAME = 'N1904'
REVISION = '3f53a5a9075cbca5c3001a000a7b004cdee251b6'
VERSION = '1.0.0'
SOURCE = ROOT / ".testdata" / "example-sources" / NAME
SOURCE.parent.mkdir(parents=True, exist_ok=True)
if not SOURCE.exists():
    subprocess.run(["git", "clone", "--filter=blob:none",
                    "https://github.com/CenterBLC/N1904.git", str(SOURCE)], check=True)
#assert not subprocess.check_output(["git", "-C", str(SOURCE), "status", "--porcelain"], text=True).strip(), "Source checkout has local changes."
subprocess.run(["git", "-C", str(SOURCE), "fetch", "origin", REVISION], check=True)
subprocess.run(["git", "-C", str(SOURCE), "checkout", "--detach", REVISION], check=True)
assert subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip() == REVISION
CORPUS = SOURCE / "tf" / VERSION
assert len(list(CORPUS.glob("*.tf"))) == 61
assert (CORPUS / "otype.tf").is_file() and (CORPUS / "oslots.tf").is_file()

## 3. Configure the one-request corpus overview

`OVERVIEW_ONLY = True` makes one OpenRouter context-analysis request from the measured N1904 inventory and the prewritten editorial guidance in `notebooks/contexts/N1904.txt`. It saves a reusable `dataset_context.json` overview for a later feature-description run. `USE_LLM = False` keeps feature descriptions off; this run does not make family-analysis, targeted-evidence, or per-feature requests.

The OpenRouter key is read from `OPENROUTER_API_KEY` or entered through a hidden prompt. It is never included in a command or notebook output. The editorial note guides audience and terminology only; corpus claims must be supported by the measured inventory.

In [ ]:
OVERVIEW_ONLY = True
USE_LLM = False  # Keep per-feature LLM descriptions off for this run.
TARGETED_EVIDENCE = False
CONTEXT_PAGES = True
PROVIDER = "openrouter"
DATASET_MODEL = 'nvidia/nemotron-3.5-super-vl:free'
FAMILY_MODEL = 'nvidia/nemotron-3.5-super-vl:free'
FEATURE_MODEL = 'nvidia/nemotron-3.5-super-vl:free'
CONTEXT = ROOT / "notebooks" / "contexts" / "N1904.txt"
CONTEXT_DIR = ROOT / ".testdata" / "context-artifacts" / 'N1904'
OUTPUT = ROOT / ".testdata" / "example-builds" / 'N1904' / ("enriched" if USE_LLM else "offline")
CACHE = ROOT / ".doc4cf-cache" / 'n1904'

if USE_LLM or OVERVIEW_ONLY:
    from getpass import getpass
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")
    assert os.environ.get("OPENROUTER_API_KEY"), "An API key is required for enrichment."

## 4. Generate and inspect the corpus overview template

The overview-only path exports the deterministic inventory, then makes one OpenRouter request to describe dataset-level conventions, ambiguities, and review needs. The prewritten N1904 note is included as editorial guidance, not as factual evidence. The request sends a compact inventory summary and caches the raw response; rerunning unchanged inputs reuses it. The notebook drops model-proposed count-comparison anomalies that are not supported by the inventory and retains an author-metadata scope question for review. No feature descriptions or feature-family analysis are requested in this stage.


In [ ]:
if OVERVIEW_ONLY:
    import hashlib
    from urllib.request import Request
    from doc4cf.context import (
        DATASET_SYSTEM, PROMPT_VERSION, _atomic_json, _object_response,
        inventory_digest, validate_dataset_context,
    )
    from doc4cf.generator import (
        PROVIDERS, export_inventory, request_with_completion_retry,
    )

    CONTEXT_DIR.mkdir(parents=True, exist_ok=True)
    inventory = export_inventory(CORPUS, CONTEXT_DIR / "inventory.json")
    inventory_hash = inventory_digest(inventory)
    inventory["inventory_sha256"] = inventory_hash
    editorial_guidance = CONTEXT.read_text(encoding="utf-8-sig")

    # Keep the request compact: preserve inventory field paths, but omit per-feature
    # examples and long metadata that are not needed for a corpus-level overview.
    overview_features = {}
    for feature_name, feature in inventory["features"].items():
        metadata = feature.get("metadata", {})
        overview_features[feature_name] = {
            "kind": feature.get("kind"),
            "metadata": {key: metadata[key] for key in (
                "description", "valueType", "title", "name", "author",
                "converters", "dataSource",
            ) if key in metadata},
            "encoding": feature.get("encoding"),
            "observed_node_types": feature.get("observed_node_types", []),
            "sources": feature.get("sources", {}),
            "targets": feature.get("targets", {}),
            "distinct_values": feature.get("distinct_values"),
            "frequent_values": feature.get("frequent_values", [])[:3],
        }
    overview_inventory = {
        "inventory_version": inventory["inventory_version"],
        "dataset": inventory["dataset"],
        "corpus_label": inventory["corpus_label"],
        "source": {"declared_version": inventory["source"].get("declared_version")},
        "structure": inventory["structure"],
        "features": overview_features,
    }
    system = DATASET_SYSTEM + (
        "\n\nEditorial guidance may shape audience and terminology, but is not "
        "evidence about the corpus. Base factual claims and evidence paths only on "
        "the measured inventory. Be concise: dataset_profile at most 140 words, "
        "no more than 6 conventions, and no more than 4 ambiguities. Do not list "
        "features one by one or infer linguistic facts from names alone."
    )
    request_data = {
        "model": DATASET_MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": json.dumps({
                "inventory": overview_inventory,
                "editorial_guidance": editorial_guidance,
            }, ensure_ascii=False, sort_keys=True)},
        ],
        "temperature": 0.1,
        "max_tokens": 6000,
        # Use JSON mode directly to avoid a schema-rejection fallback request.
        "response_format": {"type": "json_object"},
    }
    payload = json.dumps(request_data, ensure_ascii=False).encode("utf-8")
    overview_version = "n1904-overview-template-v1"
    request_hash = hashlib.sha256(
        (overview_version + "\0" + payload.decode("utf-8")).encode("utf-8")
    ).hexdigest()
    cache_file = CACHE / "context" / PROVIDER / "dataset-overview" / (request_hash + ".json")

    if cache_file.is_file():
        record = json.loads(cache_file.read_text(encoding="utf-8"))
        validate_dataset_context(record["data"])
        cache_status = "reused cached response; no OpenRouter request"
    else:
        label, key_name, endpoint = PROVIDERS[PROVIDER]
        api_key = os.environ.get(key_name)
        assert api_key, f"Set {key_name} in the environment."
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}",
        }
        request = Request(endpoint, data=payload, headers=headers)
        # Normal completion uses one request; Doc4CF retries truncation with
        # a larger max_tokens budget and returns the final response.
        reply = request_with_completion_retry(request)
        choice = reply.get("choices", [{}])[0]
        finish_reason = choice.get("finish_reason")
        if finish_reason in ("length", "content_filter"):
            print("OpenRouter finish reason:", finish_reason)
            print("OpenRouter usage:", reply.get("usage"))
            print("Response prefix:", str(choice.get("message", {}).get("content", ""))[:500])
            raise RuntimeError("OpenRouter did not return a complete corpus overview.")
        data, actual_model = _object_response(PROVIDER, reply)
        validate_dataset_context(data)
        record = {"data": data, "model": actual_model, "format_mode": "json_object"}
        _atomic_json(cache_file, record)
        cache_status = "OpenRouter response completed and cached"

    dataset_context = {
        "context_version": "1.0",
        "inventory_sha256": inventory_hash,
        "analysis": {
            "provider": PROVIDER,
            "requested_model": DATASET_MODEL,
            "responding_model": record.get("model"),
            "prompt_version": overview_version,
        },
        **record["data"],
    }
    # Conservative review of model-proposed ambiguities: raw cardinality
    # comparisons across unrelated feature domains are not evidence of errors.
    has_author_metadata = any(
        "author" in feature.get("metadata", {})
        for feature in inventory["features"].values()
    )
    dataset_context["ambiguities"] = ([
        "Feature metadata lists ‘Evangelists and apostles’ as author; the inventory does not clarify the intended scope of this field."
    ] if has_author_metadata else [])
    dataset_context["needs_review"] = bool(dataset_context["ambiguities"])
    validate_dataset_context({key: dataset_context[key] for key in (
        "dataset_profile", "conventions", "ambiguities", "needs_review",
    )})
    _atomic_json(CONTEXT_DIR / "dataset_context.json", dataset_context)
    print("LLM workload:", cache_status, "; no family or feature requests.")
    print("Response model:", record.get("model") or DATASET_MODEL)
    print("Overview saved:", CONTEXT_DIR / "dataset_context.json")
    print(json.dumps(dataset_context, ensure_ascii=False, indent=2))
elif USE_LLM:
    analysis_command = [
        sys.executable, "-m", "doc4cf", str(CORPUS),
        "--analyze-context", str(CONTEXT_DIR),
        "--provider", PROVIDER,
        "--context-model", DATASET_MODEL,
        "--family-model", FAMILY_MODEL,
        "--cache", str(CACHE),
    ]
    analysis = subprocess.run(analysis_command, cwd=ROOT, capture_output=True,
                             text=True, encoding="utf-8", errors="replace")
    print(analysis.stdout)
    print(analysis.stderr)
    if analysis.returncode:
        raise RuntimeError("Context analysis failed; review the error and rerun the cell.")
    for artifact in ("inventory.json", "dataset_context.json", "feature_families.json", "feature_relationships.json"):
        assert (CONTEXT_DIR / artifact).is_file(), artifact
    print("Context artifacts:", CONTEXT_DIR)
else:
    print("Context analysis skipped because OVERVIEW_ONLY and USE_LLM are False.")

### Review the overview template

The generated `dataset_context.json` is the reusable corpus-level overview. Review its profile, convention evidence, ambiguities, and `needs_review` flag before using it with feature descriptions. The full context workflow can later add family and relationship artifacts; this one-request run intentionally does not generate those.

In [ ]:
if OVERVIEW_ONLY:
    print((CONTEXT_DIR / "dataset_context.json").read_text(encoding="utf-8"))
elif USE_LLM:
    for artifact in ("dataset_context.json", "feature_families.json", "feature_relationships.json"):
        print("\n###", artifact)
        print((CONTEXT_DIR / artifact).read_text(encoding="utf-8"))

## 5. Leave feature generation for a later pass

This run creates only the corpus overview template. Feature-level LLM descriptions and the site build are skipped so the OpenRouter workload stays at one overview request. When you are ready to generate feature descriptions, complete the family/relationship context artifacts and run the full enrichment path; it sends the reviewed corpus context with each feature's measured evidence and the N1904 editorial note.

In [ ]:
if OVERVIEW_ONLY:
    print("Site generation skipped: this run only prepares the corpus overview template.")
else:
    command = [sys.executable, "-m", "doc4cf", str(CORPUS), str(OUTPUT),
               "--project-home", "../../index.html"]
    if USE_LLM:
        command += ["--provider", PROVIDER, "--model", FEATURE_MODEL,
                    "--feature-model", FEATURE_MODEL,
                    "--context-dir", str(CONTEXT_DIR), "--context-pages",
                    "--model-attribution", "feature", "--cache", str(CACHE)]
        if TARGETED_EVIDENCE:
            command.append("--targeted-evidence")
        if CONTEXT is not None:
            CONTEXT = Path(CONTEXT)
            assert CONTEXT.is_file(), CONTEXT
            command += ["--dataset-context", str(CONTEXT)]

    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                            encoding="utf-8", errors="replace")
    print("EXIT CODE:", result.returncode)
    print("\n--- STDOUT ---")
    print(result.stdout)
    print("\n--- STDERR ---")
    print(result.stderr)
    if result.returncode:
        raise RuntimeError("Documentation generation failed; do not publish its partial output.")
    shutil.copyfile(SOURCE / 'LICENSE.md', OUTPUT / 'DATA-LICENSE.md')
    print("Generated:", OUTPUT / "index.html")

## 6. Check the overview artifact

The saved artifact is tied to the pinned N1904 inventory hash. It can be reviewed now and combined with family/relationship artifacts before feature-level generation.

In [ ]:
if OVERVIEW_ONLY:
    overview_path = CONTEXT_DIR / "dataset_context.json"
    assert overview_path.is_file()
    assert inventory["inventory_sha256"] == dataset_context["inventory_sha256"]
    print("Verified overview template:", overview_path)
    print("needs_review:", dataset_context["needs_review"])
else:
    features = {p.stem for p in CORPUS.glob("*.tf")}
    assert {p.stem for p in (OUTPUT / "features").glob("*.html")} == features
    graph = json.loads((OUTPUT / "knowledge_graph.json").read_text(encoding="utf-8"))
    assert {"feature::" + f for f in features} <= graph["nodes"].keys()
    required_pages = ["index.html", "features.html", "structure.html", "diagram.html", "text.html"]
    if USE_LLM and CONTEXT_PAGES:
        required_pages += ["dataset.html", "relationships.html", "context-review.html", "context_review.json", "context_manifest.json"]
    for page in required_pages:
        assert (OUTPUT / page).is_file(), page
    for name in features:
        html = (OUTPUT / "features" / (name + ".html")).read_text(encoding="utf-8")
        assert ("Generated description" in html) == USE_LLM
    print(f"Verified {len(features)} feature pages and {len(required_pages)} overview/review pages.")
    print((OUTPUT / "index.html").as_uri())

## 7. Optional: publish the completed site later

The overview-only run does not generate or publish a site. Keep `PUBLISH = False` until you have generated and reviewed all feature pages and the context review report. Notebook outputs are cleared before sharing; keep credentials, context caches, and downloaded corpora out of Git.

In [ ]:
PUBLISH = False
if PUBLISH:
    assert not OVERVIEW_ONLY, "Generate and review the complete site before publishing."
    destination = ROOT / "examples" / NAME
    shutil.copytree(OUTPUT, destination, dirs_exist_ok=True)
    print("Updated:", destination)
else:
    print("Overview-only run; checked-in example unchanged.")